# Supplementary figure: Low-dimensional embedding comparison

Run after the training and evaluation commands in `bash/paper/`. SCENE outputs use the `scLDM` names referenced below.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment_scene.yaml").exists())
os.chdir(PROJECT_ROOT / "notebooks" / "figures")
for folder in ("fig_1", "fig_2", "fig_3", "fig_4", "fig_5", "fig_6", "app", "qc"):
    Path("output", folder).mkdir(parents=True, exist_ok=True)

Path("output/app/low_dim").mkdir(parents=True, exist_ok=True)


In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal


adata = ad.read_h5ad("../../data/neurips_cite_gex_stem_cells.h5ad")


def _build_palette_for_series(series):
    categories = list(pd.Categorical(series.astype(str)).categories)
    n_cat = len(categories)

    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]

    return categories, dict(zip(categories, colors))


COMMON_CATEGORIES = {}
COMMON_PALETTES = {}
for _label in ["cell_type", "label"]:
    if _label in adata.obs.columns:
        _cats, _pal = _build_palette_for_series(adata.obs[_label])
        COMMON_CATEGORIES[_label] = _cats
        COMMON_PALETTES[_label] = _pal



In [ ]:
SIZE_CELLS = 0.2
SIZE_GENES = 0.066

## SCENE

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


Z_cells = np.load('../../results/neurips_cite_stem_cells/scLDM_batch_full_3D/scLDM_cell_latent.npy')
Z_genes = np.load('../../results/neurips_cite_stem_cells/scLDM_batch_full_3D/scLDM_gene_latent.npy')


if Z_cells.shape[1] != 3 or Z_genes.shape[1] != 3:
    raise ValueError(f"Expected 3D embeddings. Got cells={Z_cells.shape}, genes={Z_genes.shape}")

def build_palette(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return categories, dict(zip(categories, colors))

labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

views = [("angle1", 20, 10), ("angle2", 20, 30), ("angle3", 30, 40), ("angle4", 10, 315)]

outdir = Path("output/app/low_dim")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette(values)

    for view_name, elev, azim in views:
        fig = plt.figure(figsize=(1.5, 1.5))
        ax = fig.add_subplot(111, projection="3d")

        for ct in categories:
            idx = values_np == ct
            if np.any(idx):
                ax.scatter(
                    Z_cells[idx, 0], Z_cells[idx, 1], Z_cells[idx, 2],
                    s=SIZE_CELLS, c=[palette[ct]], alpha=1, linewidths=0, depthshade=False, rasterized=True
                )

        ax.scatter(
            Z_genes[:, 0], Z_genes[:, 1], Z_genes[:, 2],
            s=SIZE_GENES, c="#7A1E1E", alpha=.5, linewidths=0, depthshade=False, rasterized=True 
        )

        ax.view_init(elev=elev, azim=azim)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        ax.tick_params(axis="x", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="y", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="z", which="major", labelsize=4, length=2, pad=0)

        ax.set_xlim(-5, 5)
        ax.set_ylim(-5, 5)
        ax.set_zlim(-3, 3)
        
        ax.set_position([0.02, 0.02, 0.96, 0.97])
        ax.set_box_aspect((
            np.ptp(ax.get_xlim()),
            np.ptp(ax.get_ylim()),
            np.ptp(ax.get_zlim())
        ))
        fig.subplots_adjust(left=0.0, right=1.00, bottom=0.00, top=1.0)

        # plt.savefig(
        #     outdir / f"kang_scLDM_3d_{label}_{view_name}_small.svg",
        #     bbox_inches="tight",
        #     pad_inches=0,
        #     dpi=600,
        # )
        plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import numpy as np


def build_palette_local(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)

    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]

    return categories, dict(zip(categories, colors))


Z_cells_2d = np.load('../../results/neurips_cite_stem_cells/scLDM_batch_full_2D/scLDM_cell_latent.npy')
Z_genes_2d = np.load('../../results/neurips_cite_stem_cells/scLDM_batch_full_2D/scLDM_gene_latent.npy')


if Z_cells_2d.shape[1] != 2 or Z_genes_2d.shape[1] != 2:
    raise ValueError(
        f"Expected 2D embeddings in scLDM_2d. Got cells={Z_cells_2d.shape}, genes={Z_genes_2d.shape}"
    )


labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

outdir = Path("output/app/low_dim")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette_local(values)

    # -------- 2D static (fig_3 style) --------
    fig, ax = plt.subplots(figsize=(1.5, 1.5))

    for ct in categories:
        idx = values_np == ct
        if np.any(idx):
            ax.scatter(
                Z_cells_2d[idx, 0],
                Z_cells_2d[idx, 1],
                s=SIZE_CELLS,
                c=[palette[ct]],
                alpha=1,
                linewidths=0,
                rasterized=True
            )

    ax.scatter(
        Z_genes_2d[:, 0],
        Z_genes_2d[:, 1],
        s=SIZE_GENES,
        c="#7A1E1E",
        alpha=0.5,
        linewidths=0,
        rasterized=True
    )

    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_axis_off()
    ax.set_title("")
    ax.set_box_aspect(
        np.ptp(ax.get_ylim()) / np.ptp(ax.get_xlim())
    )
    ax.set_position([0.02, 0.02, 0.96, 0.97])
    fig.subplots_adjust(left=0.0, right=1.00, bottom=0.00, top=1.0)
    # plt.savefig(
    #     outdir / f"kang_scLDM_2d_{label}_small.svg",
    #     bbox_inches="tight",
    #     pad_inches=0,
    #     dpi=600,
    # )
    plt.show()
    # plt.close(fig)


## scVI 3D

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


Z_cells = np.load('../../results/neurips_cite_stem_cells/scVI_3D/scVI_cell_latent.npy')
#Z_genes = np.load('../../results/neurips_cite_stem_cells/scVI_3D/scVI_gene_latent.npy')

def build_palette(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return categories, dict(zip(categories, colors))

labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

views = [("angle1", 20, 10), ("angle2", 20, 30), ("angle3", 30, 40), ("angle4", 10, 315)]
views = [("angle3", 30, 40)]

outdir = Path("output/app/low_dim")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette(values)

    for view_name, elev, azim in views:
        fig = plt.figure(figsize=(1.5, 1.5))
        ax = fig.add_subplot(111, projection="3d")

        for ct in categories:
            idx = values_np == ct
            if np.any(idx):
                ax.scatter(
                    Z_cells[idx, 0], Z_cells[idx, 1], Z_cells[idx, 2],
                    s=SIZE_CELLS, c=[palette[ct]], alpha=1, linewidths=0, depthshade=False, rasterized=True
                )

        ax.view_init(elev=elev, azim=azim)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        ax.tick_params(axis="x", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="y", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="z", which="major", labelsize=4, length=2, pad=0)

        ax.set_position([0.02, 0.02, 0.96, 0.97])

        fig.subplots_adjust(left=0.0, right=1.00, bottom=0.00, top=1.0)

        plt.savefig(
            outdir / f"kang_scVI_3d_{label}_{view_name}.svg",
            bbox_inches="tight",
            pad_inches=0,
            dpi=600,
        )
        plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


Z_cells = np.load('../../results/neurips_cite_stem_cells/scVI_2D/scVI_cell_latent.npy')
#Z_genes = np.load('../../results/neurips_cite_stem_cells/scVI_3D/scVI_gene_latent.npy')

def build_palette(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return categories, dict(zip(categories, colors))

labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

outdir = Path("output/app/low_dim")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette(values)

    fig = plt.figure(figsize=(1.5, 1.5))
    ax = fig.add_subplot(111,)

    for ct in categories:
        idx = values_np == ct
        if np.any(idx):
            ax.scatter(
                Z_cells[idx, 0], Z_cells[idx, 1],
                s=SIZE_CELLS, c=[palette[ct]], alpha=1, linewidths=0, rasterized=True
            )

    #ax.view_init(elev=elev, azim=azim)
    ax.tick_params(axis="x", which="major", labelsize=4, length=2, pad=0)
    ax.tick_params(axis="y", which="major", labelsize=4, length=2, pad=0)
    #ax.tick_params(axis="z", which="major", labelsize=4, length=2, pad=0)

    #ax.set_xlim(-5, 5)
    #ax.set_ylim(-5, 5)
    ax.set_axis_off()
    
    ax.set_position([0.02, 0.02, 0.96, 0.97])
    fig.subplots_adjust(left=0.0, right=1.00, bottom=0.00, top=1.0)

    plt.savefig(
        outdir / f"kang_scVI_2d.svg",
        bbox_inches="tight",
        pad_inches=0,
        dpi=600,
    )
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

Z_cells = np.load('../../results/neurips_cite_stem_cells/SIMBA_3D/SIMBA_cell_latent.npy')
Z_genes = np.load('../../results/neurips_cite_stem_cells/SIMBA_3D/SIMBA_gene_latent.npy')

if Z_cells.shape[1] != 3 or Z_genes.shape[1] != 3:
    raise ValueError(f"Expected 3D embeddings. Got cells={Z_cells.shape}, genes={Z_genes.shape}")

def build_palette(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return categories, dict(zip(categories, colors))

labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

views = [("angle1", 20, 10), ("angle2", 20, 30), ("angle3", 30, 40), ("angle4", 10, 315)]
views = [
    ("angle1", 20, 10),
    ("angle2", 20, 30),
    ("angle3", 30, 40),
    ("angle4", 10, 315),

    # more options
    ("angle5", 25, 60),
    ("angle6", 25, 90),
    ("angle7", 20, 120),
    ("angle8", 30, 150),
    ("angle9", 15, 210),
    ("angle10", 25, 270),
    ("angle11", 45, 330),
    ("angle12", 60, 45),
]

    
views = [("angle6", 25, 105),]

outdir = Path("output/app/low_dim")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette(values)

    for view_name, elev, azim in views:
        print(f"Plotting {label} - {view_name} (elev={elev}, azim={azim})")
        fig = plt.figure(figsize=(1.5, 1.5))
        ax = fig.add_subplot(111, projection="3d")

        for ct in categories:
            idx = values_np == ct
            if np.any(idx):
                ax.scatter(
                    Z_cells[idx, 0], Z_cells[idx, 1], Z_cells[idx, 2],
                    s=SIZE_CELLS, c=[palette[ct]], alpha=1, linewidths=0, depthshade=False, rasterized=True
                )

        ax.scatter(
            Z_genes[:, 0], Z_genes[:, 1], Z_genes[:, 2],
            s=SIZE_GENES, c="#7A1E1E", alpha=.5, linewidths=0, depthshade=False, rasterized=True
        )

        ax.view_init(elev=elev, azim=azim)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        ax.tick_params(axis="x", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="y", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="z", which="major", labelsize=4, length=2, pad=0)

        ax.set_xlim(-5, 5)
        ax.set_ylim(-5, 5)
        ax.set_zlim(-3, 2)
        
        ax.set_position([0.02, 0.02, 0.96, 0.97])
        ax.set_box_aspect((
            np.ptp(ax.get_xlim()),
            np.ptp(ax.get_ylim()),
            np.ptp(ax.get_zlim())
        ))
        fig.subplots_adjust(left=0.0, right=1.00, bottom=0.00, top=1.0)

        plt.savefig(
            outdir / f"kang_SIMBA_3d_{label}_{view_name}.svg",
            bbox_inches="tight",
            pad_inches=0,
            dpi=600,
        )
        plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


Z_cells = np.load('../../results/neurips_cite_stem_cells/SIMBA_2D/SIMBA_cell_latent.npy')
Z_genes = np.load('../../results/neurips_cite_stem_cells/SIMBA_2D/SIMBA_gene_latent.npy')


def build_palette(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return categories, dict(zip(categories, colors))

labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

views = [("angle1", 20, 10), ("angle2", 20, 30), ("angle3", 30, 40), ("angle4", 10, 315)]

outdir = Path("output/app/low_dim")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette(values)

    fig = plt.figure(figsize=(1.5, 1.5))
    ax = fig.add_subplot(111,)


    for ct in categories:
        idx = values_np == ct
        if np.any(idx):
            ax.scatter(
                Z_cells[idx, 0], Z_cells[idx, 1],
                s=SIZE_CELLS, c=[palette[ct]], alpha=1, linewidths=0
            )

    ax.scatter(
            Z_genes[:, 0], Z_genes[:, 1],
            s=SIZE_GENES, c="#7A1E1E", alpha=.5, linewidths=0
        )

    #ax.view_init(elev=elev, azim=azim)
    ax.tick_params(axis="x", which="major", labelsize=4, length=2, pad=0)
    ax.tick_params(axis="y", which="major", labelsize=4, length=2, pad=0)
    #ax.tick_params(axis="z", which="major", labelsize=4, length=2, pad=0)

    ax.set_xlim(-7, 3)
    ax.set_ylim(-4, 2.5)
    ax.set_axis_off()
    ax.set_position([0.02, 0.02, 0.96, 0.97])
    ax.set_box_aspect(
        np.ptp(ax.get_ylim()) / np.ptp(ax.get_xlim())
    )
    
    fig.subplots_adjust(left=0.0, right=1.00, bottom=0.00, top=1.0)

    plt.savefig(
        outdir / f"kang_SIMBA_2d.png",
        bbox_inches="tight",
        pad_inches=0,
        dpi=600,
    )
    plt.show()


## PCA

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


Z_cells = np.load('../../results/neurips_cite_stem_cells/PCA_3D/X_pca_harmony_cell_latent.npy')

def build_palette(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return categories, dict(zip(categories, colors))

labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

views = [("angle1", 20, 10), ("angle2", 20, 30), ("angle3", 30, 40), ("angle4", 10, 315)]
views = [ ("angle3", 30, 40),]

outdir = Path("output/app/low_dim")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette(values)

    for view_name, elev, azim in views:
        fig = plt.figure(figsize=(1.5, 1.5))
        ax = fig.add_subplot(111, projection="3d")

        for ct in categories:
            idx = values_np == ct
            if np.any(idx):
                ax.scatter(
                    Z_cells[idx, 0], Z_cells[idx, 1], Z_cells[idx, 2],
                    s=SIZE_CELLS, c=[palette[ct]], alpha=1, linewidths=0, depthshade=False, rasterized=True
                )

        ax.view_init(elev=elev, azim=azim)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        ax.tick_params(axis="x", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="y", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="z", which="major", labelsize=4, length=2, pad=0)

        # ax.set_xlim(-5, 5)
        # ax.set_ylim(-5, 5)
        # ax.set_zlim(-3, 3)
        
        ax.set_position([0.02, 0.02, 0.96, 0.97])
        fig.subplots_adjust(left=0.0, right=1.00, bottom=0.00, top=1.0)

        plt.savefig(
            outdir / f"kang_PCA_3d.svg",
            bbox_inches="tight",
            pad_inches=0,
            dpi=600,
        )
        plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


Z_cells = np.load('../../results/neurips_cite_stem_cells/PCA_2D/X_pca_harmony_cell_latent.npy')
#Z_genes = np.load('../../results/neurips_cite_stem_cells/scVI_3D/scVI_gene_latent.npy')

def build_palette(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return categories, dict(zip(categories, colors))

labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

views = [("angle1", 20, 10), ("angle2", 20, 30), ("angle3", 30, 40), ("angle4", 10, 315)]

outdir = Path("output/app/low_dim")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette(values)

    fig = plt.figure(figsize=(1.5, 1.5))
    ax = fig.add_subplot(111,)

    for ct in categories:
        idx = values_np == ct
        if np.any(idx):
            ax.scatter(
                Z_cells[idx, 0], Z_cells[idx, 1],
                s=SIZE_CELLS, c=[palette[ct]], alpha=1, linewidths=0, rasterized=True
            )

    #ax.view_init(elev=elev, azim=azim)
    ax.set_axis_off()

    ax.tick_params(axis="x", which="major", labelsize=4, length=2, pad=0)
    ax.tick_params(axis="y", which="major", labelsize=4, length=2, pad=0)
    #ax.tick_params(axis="z", which="major", labelsize=4, length=2, pad=0)

    #ax.set_xlim(-5, 5)
    #ax.set_ylim(-5, 5)
    
    ax.set_position([0.02, 0.02, 0.96, 0.97])
    fig.subplots_adjust(left=0.0, right=1.00, bottom=0.00, top=1.0)

    plt.savefig(
        outdir / f"kang_PCA_2d.svg",
        bbox_inches="tight",
        pad_inches=0,
        dpi=600,
    )
    plt.show()
